In [ ]:
import duckdb
import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display

path = r"C:\Users\Pattanun\OneDrive\Desktop\SEMTM0044_2025_AYEAR_TEAM34\notebooks\03_Mew"

# ---- download options ----
print("Loading options...")

valid_mut_genes = duckdb.sql(f"""
    SELECT DISTINCT hugo_symbol 
    FROM '{path}/fact_mutations.csv'
    ORDER BY hugo_symbol
""").df()["hugo_symbol"].tolist()

valid_fusion_genes = duckdb.sql(f"""
    SELECT gene1_hugo as gene FROM '{path}/fact_fusions.csv'
    UNION
    SELECT gene2_hugo as gene FROM '{path}/fact_fusions.csv'
    ORDER BY gene
""").df()["gene"].tolist()

lineage_options = ["all"] + duckdb.sql(f"""
    SELECT DISTINCT lineage 
    FROM '{path}/dim_cell_lines.csv'
    WHERE lineage IS NOT NULL
    ORDER BY lineage
""").df()["lineage"].tolist()

all_diseases = duckdb.sql(f"""
    SELECT DISTINCT primary_disease
    FROM '{path}/dim_cell_lines.csv'
    WHERE primary_disease IS NOT NULL
    ORDER BY primary_disease
""").df()["primary_disease"].tolist()

metabolite_cols = [c for c in duckdb.sql(f"""
    SELECT * FROM '{path}/metabolomics_clean.parquet' LIMIT 1
""").df().columns.tolist() if c not in ["CCLE_ID", "DepMap_ID"]]

print(f"Ready. {len(valid_mut_genes)} mutation genes | {len(valid_fusion_genes)} fusion genes | {len(lineage_options)-1} lineages | {len(all_diseases)} diseases | {len(metabolite_cols)} metabolites")

# ---- global result ----
df_result          = None
hugo_symbol_global = None

# ---- normalize helper ----
def normalize(series):
    s = series.fillna(series.mean())
    if s.max() == s.min():
        return pd.Series(0, index=s.index)
    return (s - s.min()) / (s.max() - s.min())

# ---- Widget UI ----
target_gene_input = widgets.Combobox(
    value="ENSG00000146648",
    options=[],
    description="Target gene:",
    ensure_option=False,
    style={"description_width": "130px"},
    layout=widgets.Layout(width="420px")
)

use_mutation_input = widgets.Checkbox(
    value=False,
    description="Use mutation filter",
    style={"description_width": "130px"}
)

mutation_gene_input = widgets.Combobox(
    value="KRAS",
    options=valid_mut_genes,
    description="Mutation gene:",
    ensure_option=False,
    style={"description_width": "130px"},
    layout=widgets.Layout(width="420px")
)

mutation_mode_input = widgets.ToggleButtons(
    options=["exclude", "include"],
    value="exclude",
    description="Mode:",
    style={"description_width": "130px"}
)

mutation_type_input = widgets.Dropdown(
    options=["damaging", "hotspot", "driver", "all"],
    value="damaging",
    description="Mutation type:",
    style={"description_width": "130px"},
    layout=widgets.Layout(width="320px")
)

use_fusion_input = widgets.Checkbox(
    value=False,
    description="Use fusion filter",
    style={"description_width": "130px"}
)

fusion_gene_input = widgets.Combobox(
    value="",
    options=valid_fusion_genes,
    description="Fusion gene:",
    ensure_option=False,
    placeholder="e.g. EGFR",
    style={"description_width": "130px"},
    layout=widgets.Layout(width="420px")
)

fusion_mode_input = widgets.ToggleButtons(
    options=["exclude", "include"],
    value="exclude",
    description="Mode:",
    style={"description_width": "130px"}
)

fusion_confidence_input = widgets.Dropdown(
    options=["all", "high", "medium", "low"],
    value="high",
    description="Confidence:",
    style={"description_width": "130px"},
    layout=widgets.Layout(width="320px")
)

lineage_input = widgets.Dropdown(
    options=lineage_options,
    value="all",
    description="Lineage:",
    style={"description_width": "130px"},
    layout=widgets.Layout(width="320px")
)

disease_input = widgets.Dropdown(
    options=["all"] + all_diseases,
    value="all",
    description="Disease:",
    style={"description_width": "130px"},
    layout=widgets.Layout(width="320px")
)

def on_lineage_change(change):
    selected_lineage = change["new"]
    if selected_lineage == "all":
        new_diseases = ["all"] + all_diseases
    else:
        new_diseases = ["all"] + duckdb.sql(f"""
            SELECT DISTINCT primary_disease
            FROM '{path}/dim_cell_lines.csv'
            WHERE lineage = '{selected_lineage}'
            AND primary_disease IS NOT NULL
            ORDER BY primary_disease
        """).df()["primary_disease"].tolist()
    disease_input.options = new_diseases
    disease_input.value = "all"

lineage_input.observe(on_lineage_change, names="value")

metabolite_input = widgets.Combobox(
    value="",
    options=metabolite_cols,
    description="Metabolite:",
    ensure_option=False,
    placeholder="optional e.g. aconitate",
    style={"description_width": "130px"},
    layout=widgets.Layout(width="420px")
)

top_n_input = widgets.IntSlider(
    value=10,
    min=5,
    max=50,
    step=5,
    description="Top N:",
    style={"description_width": "130px"},
    layout=widgets.Layout(width="420px")
)

sort_by_input = widgets.Dropdown(
    options=[
        ("final_score (all 3 sources, RNA+protein)", "final_score"),
        ("HPA confidence (RNA+protein)", "HPA_confidence"),
        ("DepMap confidence (RNA+protein)", "DepMap_confidence"),
        ("GEO confidence (RNA+protein)", "GEO_confidence"),
        ("HPA percentile (RNA only)", "hpa_score"),
        ("DepMap percentile (RNA only)", "depmap_score"),
        ("GEO percentile (RNA only)", "geo_score"),
    ],
    value="final_score",
    description="Sort by:",
    style={"description_width": "130px"},
    layout=widgets.Layout(width="420px")
)

show_cvcl_input = widgets.Checkbox(
    value=False,
    description="CVCL_ID",
    style={"description_width": "150px"}
)

show_growth_input = widgets.Checkbox(
    value=False,
    description="Growth pattern",
    style={"description_width": "150px"}
)

show_metastasis_input = widgets.Checkbox(
    value=False,
    description="Primary/Metastasis",
    style={"description_width": "150px"}
)

show_collection_input = widgets.Checkbox(
    value=False,
    description="Collection site",
    style={"description_width": "150px"}
)

show_subtype_input = widgets.Checkbox(
    value=False,
    description="Subtype details",
    style={"description_width": "150px"}
)

show_sex_age_input = widgets.Checkbox(
    value=False,
    description="Sex / Age",
    style={"description_width": "150px"}
)

show_dev_input = widgets.Checkbox(
    value=False,
    description="Show debug columns",
    style={"description_width": "130px"}
)

run_button = widgets.Button(
    description="Run ranking",
    button_style="primary",
    layout=widgets.Layout(width="200px")
)

log_output   = widgets.Output()
table_output = widgets.Output()

mutation_box = widgets.VBox([
    mutation_gene_input,
    mutation_mode_input,
    mutation_type_input
])

fusion_box = widgets.VBox([
    fusion_gene_input,
    fusion_mode_input,
    fusion_confidence_input
])

def toggle_mutation(change):
    mutation_box.layout.display = "" if change["new"] else "none"

def toggle_fusion(change):
    fusion_box.layout.display = "" if change["new"] else "none"

use_mutation_input.observe(toggle_mutation, names="value")
use_fusion_input.observe(toggle_fusion, names="value")

mutation_box.layout.display = "none"
fusion_box.layout.display   = "none"

# ---- display result function ----
def display_result(change=None):
    global df_result, hugo_symbol_global
    if df_result is None:
        return

    top_n       = top_n_input.value
    hugo_symbol = hugo_symbol_global or ""
    sort_col    = sort_by_input.value
    # map percentile-table column -> equivalent z-score/min-max-table column
    SORT_COL_TO_ZSCORE = {
        "final_score":       "final_score_zscore",
        "HPA_confidence":    "HPA_confidence_zscore",
        "DepMap_confidence": "DepMap_confidence_zscore",
        "GEO_confidence":    "GEO_confidence_zscore",
        "hpa_score":         "hpa_zscore",
        "depmap_score":      "depmap_zscore",
        "geo_score":         "geo_zscore",
    }
    sort_col_z  = SORT_COL_TO_ZSCORE[sort_col]

    prod_cols = ["ACH_ID", "cell_line_name", "primary_disease", "lineage",
                 "hpa_score", "depmap_score", "geo_score", "protein_score",
                 "HPA_confidence", "DepMap_confidence", "GEO_confidence", "final_score"]

    if show_cvcl_input.value:
        prod_cols.insert(1, "CVCL_ID")
    if show_growth_input.value:
        prod_cols.append("growth_pattern")
    if show_metastasis_input.value:
        prod_cols.append("primary_or_metastasis")
    if show_collection_input.value:
        prod_cols.append("sample_collection_site")
    if show_subtype_input.value:
        prod_cols.append("lineage_subtype")
        prod_cols.append("Subtype")
    if show_sex_age_input.value:
        prod_cols.append("sex")
        prod_cols.append("age")

    metab_col = metabolite_input.value.strip()
    if metab_col and metab_col in df_result.columns:
        prod_cols.append(metab_col)

    if "fusion_events" in df_result.columns:
        prod_cols += ["fusion_events", "max_ffpm"]

    if show_dev_input.value:
        debug_cols = [c for c in df_result.columns if "_mutations" in c or "_has_damaging" in c]
        prod_cols += debug_cols

    show_cols = [c for c in prod_cols if c in df_result.columns]

    display_df = df_result.sort_values(sort_col, ascending=False).head(top_n)[show_cols].copy()
    display_df = display_df.rename(columns={
        "hpa_score": "hpa_percentile",
        "depmap_score": "depmap_percentile",
        "geo_score": "geo_percentile",
        "protein_score": "protein_percentile",
        "HPA_confidence": "hpa_score",
        "DepMap_confidence": "depmap_score",
        "GEO_confidence": "geo_score",
    })
    display_df = display_df.fillna("No information")
    display_df = display_df.replace("unknown", "No information")
    display_df = display_df.replace("None", "No information")

    # ---- second table: same structure as table 1, z-score/min-max version for comparison ----
    zscore_cols = ["ACH_ID", "cell_line_name", "primary_disease", "lineage",
                   "hpa_zscore", "depmap_zscore", "geo_zscore", "protein_zscore",
                   "HPA_confidence_zscore", "DepMap_confidence_zscore", "GEO_confidence_zscore",
                   "final_score_zscore"]
    zscore_cols = [c for c in zscore_cols if c in df_result.columns]
    display_df_z = df_result.sort_values(sort_col_z, ascending=False).head(top_n)[zscore_cols].copy()
    display_df_z = display_df_z.rename(columns={
        "HPA_confidence_zscore": "hpa_score",
        "DepMap_confidence_zscore": "depmap_score",
        "GEO_confidence_zscore": "geo_score",
        "final_score_zscore": "final_score",
    })
    display_df_z = display_df_z.fillna("No information")

    with table_output:
        table_output.clear_output()
        print(f"\n=== Top {top_n} ({hugo_symbol}) — PERCENTILE method, sorted by {sort_col} ===")
        print(display_df.to_string(index=False))
        print(f"\n=== Top {top_n} ({hugo_symbol}) — Z-SCORE / MIN-MAX method, sorted by {sort_col_z} (for comparison) ===")
        print(display_df_z.to_string(index=False))

show_cvcl_input.observe(display_result, names="value")
show_growth_input.observe(display_result, names="value")
show_metastasis_input.observe(display_result, names="value")
show_collection_input.observe(display_result, names="value")
show_subtype_input.observe(display_result, names="value")
show_sex_age_input.observe(display_result, names="value")
show_dev_input.observe(display_result, names="value")
top_n_input.observe(display_result, names="value")
sort_by_input.observe(display_result, names="value")

display(
    widgets.VBox([
        target_gene_input,
        widgets.HTML("<b>— Mutation Filter —</b>"),
        use_mutation_input,
        mutation_box,
        widgets.HTML("<b>— Fusion Filter —</b>"),
        use_fusion_input,
        fusion_box,
        widgets.HTML("<b>— Cell Line Filter —</b>"),
        lineage_input,
        disease_input,
        widgets.HTML("<b>— Other Options —</b>"),
        metabolite_input,
        top_n_input,
        sort_by_input,
        widgets.HTML("<b>— Show Columns —</b>"),
        show_cvcl_input,
        show_growth_input,
        show_metastasis_input,
        show_collection_input,
        show_subtype_input,
        show_sex_age_input,
        show_dev_input,
        run_button,
        log_output,
        table_output
    ])
)

# ---- Run on button click ----
def run_ranking(b):
    global df_result, hugo_symbol_global

    with log_output:
        log_output.clear_output()

        target_gene       = target_gene_input.value.strip()
        use_mutation      = use_mutation_input.value
        mutation_gene     = mutation_gene_input.value.strip()
        mutation_mode     = mutation_mode_input.value
        mutation_type     = mutation_type_input.value
        use_fusion        = use_fusion_input.value
        fusion_gene       = fusion_gene_input.value.strip()
        fusion_mode       = fusion_mode_input.value
        fusion_confidence = fusion_confidence_input.value
        lineage_filter    = lineage_input.value
        disease_filter    = disease_input.value
        metabolite        = metabolite_input.value.strip()

        # Validate
        gene_exists = duckdb.sql(f"""
            SELECT COUNT(*) as n FROM '{path}/depmap.csv'
            WHERE ensembl_id = '{target_gene}'
            LIMIT 1
        """).df()["n"].values[0]

        if gene_exists == 0:
            print(f"Error: '{target_gene}' not found.")
            return

        if use_mutation and mutation_gene and mutation_gene not in valid_mut_genes:
            print(f"Error: '{mutation_gene}' not found in mutation data.")
            return

        if use_fusion and fusion_gene and fusion_gene not in valid_fusion_genes:
            print(f"Error: '{fusion_gene}' not found in fusion data.")
            return

        if metabolite and metabolite not in metabolite_cols:
            print(f"Error: '{metabolite}' not found in metabolomics data.")
            return

        # ดึง hugo_symbol จาก dim_genes
        gene_info = duckdb.sql(f"""
            SELECT hugo_symbol, gene_type
            FROM '{path}/dim_genes.csv'
            WHERE ensembl_id = '{target_gene}'
            LIMIT 1
        """).df()

        hugo_symbol        = gene_info["hugo_symbol"].values[0] if len(gene_info) > 0 else target_gene
        gene_type          = gene_info["gene_type"].values[0] if len(gene_info) > 0 else "unknown"
        hugo_symbol_global = hugo_symbol

        print(f"Running... target={target_gene} ({hugo_symbol}) | gene_type={gene_type} | mutation={'on' if use_mutation else 'off'} | fusion={'on' if use_fusion else 'off'} | lineage={lineage_filter} | disease={disease_filter}")

        # Step 1 — query + z_score (PARTITIONED BY source — FIX) + pivot
        pivot = duckdb.sql(f"""
            WITH combined AS (
                SELECT ACH_ID, CVCL_ID, ensembl_id, tpm, 'hpa' as source
                FROM '{path}/hpa.csv'
                WHERE ensembl_id = '{target_gene}'
                UNION ALL
                SELECT ACH_ID, CVCL_ID, ensembl_id, tpm, 'depmap' as source
                FROM '{path}/depmap.csv'
                WHERE ensembl_id = '{target_gene}'
                UNION ALL
                SELECT ACH_ID, CVCL_ID, ensembl_id, AVG(tpm) as tpm, 'geo' as source
                FROM '{path}/geo.csv'
                WHERE ensembl_id = '{target_gene}'
                GROUP BY ACH_ID, CVCL_ID, ensembl_id
            ),
            with_z AS (
                SELECT *,
                    (tpm - AVG(tpm) OVER (PARTITION BY source)) /
                    NULLIF(STDDEV(tpm) OVER (PARTITION BY source), 0) as z_score
                FROM combined
            )
            PIVOT with_z ON source USING AVG(z_score)
            GROUP BY ACH_ID, CVCL_ID, ensembl_id
        """).df()

        # Step 2 — RNA_z, n_sources, source_std, sources_available
        pivot["RNA_z"] = pivot[["hpa", "depmap", "geo"]].mean(axis=1, skipna=True)
        pivot["n_sources"] = pivot[["hpa", "depmap", "geo"]].notna().sum(axis=1)
        pivot["source_std"] = pivot[["hpa", "depmap", "geo"]].std(axis=1, skipna=True)
        pivot["sources_available"] = pivot.apply(
            lambda row: ", ".join([
                s for s, col in [("HPA", "hpa"), ("DepMap", "depmap"), ("GEO", "geo")]
                if pd.notna(row[col])
            ]), axis=1
        )

        # Step 3 — protein_intensity
        prot = duckdb.sql(f"""
            SELECT ACH_ID, protein_intensity
            FROM '{path}/proteomics.csv'
            WHERE ensembl_id = '{target_gene}'
        """).df()

        df = pivot.merge(prot, on="ACH_ID", how="left")
        df_clean = df[df["ACH_ID"].notna()].copy()
        df_clean["protein_intensity_original"] = df_clean["protein_intensity"].copy()

        # Step 4 — Mutation filter
        mut_condition = None
        if use_mutation and mutation_gene:
            mut_condition = {
                "damaging": "is_damaging = True",
                "hotspot":  "is_hotspot = True",
                "driver":   "is_driver = True",
                "all":      "1=1"
            }[mutation_type]

            mut_filter = duckdb.sql(f"""
                SELECT DISTINCT ach_id
                FROM '{path}/fact_mutations.csv'
                WHERE hugo_symbol = '{mutation_gene}'
                AND {mut_condition}
            """).df()

            print(f"Cell lines with {mutation_gene} ({mutation_type}): {len(mut_filter)}")

            if mutation_mode == "exclude":
                df_clean = df_clean[~df_clean["ACH_ID"].isin(mut_filter["ach_id"])].copy()
            elif mutation_mode == "include":
                df_clean = df_clean[df_clean["ACH_ID"].isin(mut_filter["ach_id"])].copy()

        # Step 5 — Fusion filter
        if use_fusion and fusion_gene:
            fusion_conf_condition = "" if fusion_confidence == "all" else f"AND confidence = '{fusion_confidence}'"

            fusion_filter = duckdb.sql(f"""
                SELECT DISTINCT ach_id
                FROM '{path}/fact_fusions.csv'
                WHERE (gene1_hugo = '{fusion_gene}' OR gene2_hugo = '{fusion_gene}')
                {fusion_conf_condition}
            """).df()

            print(f"Cell lines with {fusion_gene} fusion ({fusion_confidence}): {len(fusion_filter)}")

            if fusion_mode == "exclude":
                df_clean = df_clean[~df_clean["ACH_ID"].isin(fusion_filter["ach_id"])].copy()
            elif fusion_mode == "include":
                df_clean = df_clean[df_clean["ACH_ID"].isin(fusion_filter["ach_id"])].copy()

        # Step 6 — join dim_cell_lines
        dim = duckdb.sql(f"""
            SELECT ach_id, cvcl_id, cell_line_name, primary_disease, Subtype,
                   lineage, lineage_subtype,
                   growth_pattern, primary_or_metastasis,
                   sample_collection_site, sex, age
            FROM '{path}/dim_cell_lines.csv'
        """).df()

        df_clean = df_clean.merge(dim, left_on="ACH_ID", right_on="ach_id", how="left")

        # Step 7 — Lineage + Disease filter
        if lineage_filter != "all":
            df_clean = df_clean[df_clean["lineage"] == lineage_filter].copy()

        if disease_filter != "all":
            df_clean = df_clean[df_clean["primary_disease"] == disease_filter].copy()

        print(f"Cell lines after filter: {len(df_clean)}")

        if len(df_clean) == 0:
            print("No cell lines found. Please adjust your criteria.")
            return

        # Step 8 — Metabolomics (optional)
        if metabolite:
            metab = duckdb.sql(f"""
                SELECT "DepMap_ID", "{metabolite}" as metabolite_value
                FROM '{path}/metabolomics_clean.parquet'
            """).df().rename(columns={"DepMap_ID": "ACH_ID"})
            df_clean = df_clean.merge(metab, on="ACH_ID", how="left")
            df_clean = df_clean.rename(columns={"metabolite_value": metabolite})

        # Step 9 — Fusion info (ถ้า include)
        if use_fusion and fusion_gene and fusion_mode == "include":
            fusion_conf_condition = "" if fusion_confidence == "all" else f"AND confidence = '{fusion_confidence}'"

            fusion_info = duckdb.sql(f"""
                SELECT ach_id,
                       STRING_AGG(fusion_name || ' (' || confidence || ')', ', ') as fusion_events,
                       MAX(ffpm) as max_ffpm
                FROM '{path}/fact_fusions.csv'
                WHERE (gene1_hugo = '{fusion_gene}' OR gene2_hugo = '{fusion_gene}')
                {fusion_conf_condition}
                GROUP BY ach_id
            """).df().rename(columns={"ach_id": "ACH_ID"})

            df_clean = df_clean.merge(fusion_info, on="ACH_ID", how="left")

        # Step 10 — Per-source scores (PERCENTILE, not min-max) + per-source confidence (source + protein)
        # percentile is invariant to monotonic transforms, so ranking directly on the
        # z-score columns (hpa/depmap/geo) gives the same percentile as ranking on raw tpm —
        # no need to re-fetch raw tpm.

        df_clean["hpa_score"] = np.where(
            df_clean["hpa"].notna(),
            df_clean["hpa"].rank(pct=True),
            np.nan
        )
        df_clean["depmap_score"] = np.where(
            df_clean["depmap"].notna(),
            df_clean["depmap"].rank(pct=True),
            np.nan
        )
        df_clean["geo_score"] = np.where(
            df_clean["geo"].notna(),
            df_clean["geo"].rank(pct=True),
            np.nan
        )

        # protein score — percentile if available, NaN if not
        df_clean["protein_score"] = np.where(
            df_clean["protein_intensity_original"].notna(),
            df_clean["protein_intensity_original"].rank(pct=True),
            np.nan
        )

        # ---- per-source confidence = source_score + protein_score ----
        # Weights: RNA and protein treated as equally important (no biological
        # evidence to favor one over the other for this use case)
        SOURCE_WEIGHT    = 0.5
        PROTEIN_WEIGHT   = 0.5
        PENALTY_STRENGTH = 0.5   # how much of the missing weight actually gets deducted
        MISSING_PENALTY  = 1 - PROTEIN_WEIGHT * PENALTY_STRENGTH   # = 0.75 with values above

        def combine_with_protein(source_score, protein_score):
            # source_score is assumed notna() here (RNA gatekeeper checked by caller)
            return np.where(
                protein_score.notna(),
                source_score * SOURCE_WEIGHT + protein_score * PROTEIN_WEIGHT,
                source_score * MISSING_PENALTY
            )

        # Case 1 & 2: RNA present (gatekeeper) -> confidence computed; NaN protein handled inside
        df_clean["HPA_confidence"] = np.where(
            df_clean["hpa_score"].notna(),
            combine_with_protein(df_clean["hpa_score"], df_clean["protein_score"]),
            np.nan
        )
        df_clean["DepMap_confidence"] = np.where(
            df_clean["depmap_score"].notna(),
            combine_with_protein(df_clean["depmap_score"], df_clean["protein_score"]),
            np.nan
        )
        df_clean["GEO_confidence"] = np.where(
            df_clean["geo_score"].notna(),
            combine_with_protein(df_clean["geo_score"], df_clean["protein_score"]),
            np.nan
        )

        # Case 3: no RNA in ANY source, but protein exists -> kept separate,
        # NOT folded into HPA/DepMap/GEO_confidence (would duplicate one value
        # across all 3 "independent" sources and inflate rank agreement).
        no_rna_at_all = (
            df_clean["hpa_score"].isna() &
            df_clean["depmap_score"].isna() &
            df_clean["geo_score"].isna()
        )
        df_clean["Protein_only_confidence"] = np.where(
            no_rna_at_all & df_clean["protein_score"].notna(),
            df_clean["protein_score"] * MISSING_PENALTY,
            np.nan
        )

        # ---- Z-SCORE / MIN-MAX version (kept in parallel, for comparison table only) ----
        # min-max of the z-score columns == min-max of raw tpm (proven equivalent),
        # sensitive to outliers/skew — shown side-by-side with the percentile method above.
        def normalize_minmax(col):
            s = df_clean[col].copy()
            s_min, s_max = s.min(), s.max()
            if s_max == s_min:
                return pd.Series(0, index=s.index)
            return (s - s_min) / (s_max - s_min)

        df_clean["hpa_zscore"] = np.where(df_clean["hpa"].notna(), normalize_minmax("hpa"), np.nan)
        df_clean["depmap_zscore"] = np.where(df_clean["depmap"].notna(), normalize_minmax("depmap"), np.nan)
        df_clean["geo_zscore"] = np.where(df_clean["geo"].notna(), normalize_minmax("geo"), np.nan)
        df_clean["protein_zscore"] = np.where(
            df_clean["protein_intensity_original"].notna(),
            normalize_minmax("protein_intensity_original"),
            np.nan
        )

        df_clean["HPA_confidence_zscore"] = np.where(
            df_clean["hpa_zscore"].notna(),
            combine_with_protein(df_clean["hpa_zscore"], df_clean["protein_zscore"]),
            np.nan
        )
        df_clean["DepMap_confidence_zscore"] = np.where(
            df_clean["depmap_zscore"].notna(),
            combine_with_protein(df_clean["depmap_zscore"], df_clean["protein_zscore"]),
            np.nan
        )
        df_clean["GEO_confidence_zscore"] = np.where(
            df_clean["geo_zscore"].notna(),
            combine_with_protein(df_clean["geo_zscore"], df_clean["protein_zscore"]),
            np.nan
        )
        df_clean["hpa_zscore"] = df_clean["hpa_zscore"].round(3)
        df_clean["depmap_zscore"] = df_clean["depmap_zscore"].round(3)
        df_clean["geo_zscore"] = df_clean["geo_zscore"].round(3)
        df_clean["protein_zscore"] = df_clean["protein_zscore"].round(3)
        df_clean["HPA_confidence_zscore"] = df_clean["HPA_confidence_zscore"].round(3)
        df_clean["DepMap_confidence_zscore"] = df_clean["DepMap_confidence_zscore"].round(3)
        df_clean["GEO_confidence_zscore"] = df_clean["GEO_confidence_zscore"].round(3)

        # ---- old combined "confidence" (mean-across-source) removed — superseded
        # by rank aggregation below, which avoids combining raw magnitudes
        # across sources with different platforms/distributions ----

        df_clean["HPA_confidence"]              = df_clean["HPA_confidence"].round(3)
        df_clean["DepMap_confidence"]            = df_clean["DepMap_confidence"].round(3)
        df_clean["GEO_confidence"]               = df_clean["GEO_confidence"].round(3)
        df_clean["Protein_only_confidence"]      = df_clean["Protein_only_confidence"].round(3)

        # Step 11 — Rank Aggregation (per-source rank, then average — NOT combining raw magnitudes)
        # Rank each cell line within each source's confidence (1 = best). Missing source -> NaN rank.
        df_clean["HPA_rank"] = df_clean["HPA_confidence"].rank(ascending=False, method="min")
        df_clean["DepMap_rank"] = df_clean["DepMap_confidence"].rank(ascending=False, method="min")
        df_clean["GEO_rank"] = df_clean["GEO_confidence"].rank(ascending=False, method="min")

        df_clean["avg_rank"] = df_clean[["HPA_rank", "DepMap_rank", "GEO_rank"]].mean(axis=1, skipna=True)

        # Final rank = position after sorting by avg_rank ascending (lower avg_rank = better)
        df_clean["rank"] = df_clean["avg_rank"].rank(ascending=True, method="min").astype(int)

        # ---- final_score: mean of the 3 confidence values that ARE present ----
        # (skipna mean -> divides by however many sources have data: /1 if only one,
        # /2 if two, /3 if all three — not a fixed /3 that dilutes cell lines with
        # fewer sources)
        df_clean["final_score"] = df_clean[
            ["HPA_confidence", "DepMap_confidence", "GEO_confidence"]
        ].mean(axis=1, skipna=True).round(3)

        # ---- same rank aggregation, but using the z-score/min-max confidence
        # (independent ranking, so the comparison table can show its own top N,
        # not just the percentile method's top N re-scored) ----
        df_clean["HPA_rank_zscore"] = df_clean["HPA_confidence_zscore"].rank(ascending=False, method="min")
        df_clean["DepMap_rank_zscore"] = df_clean["DepMap_confidence_zscore"].rank(ascending=False, method="min")
        df_clean["GEO_rank_zscore"] = df_clean["GEO_confidence_zscore"].rank(ascending=False, method="min")
        df_clean["avg_rank_zscore"] = df_clean[["HPA_rank_zscore", "DepMap_rank_zscore", "GEO_rank_zscore"]].mean(axis=1, skipna=True)
        df_clean["rank_zscore"] = df_clean["avg_rank_zscore"].rank(ascending=True, method="min").astype(int)

        # same "mean of what's present" final_score, but for the z-score/min-max confidence
        df_clean["final_score_zscore"] = df_clean[
            ["HPA_confidence_zscore", "DepMap_confidence_zscore", "GEO_confidence_zscore"]
        ].mean(axis=1, skipna=True).round(3)

        # Step 12 — debug columns
        if show_dev_input.value and use_mutation and mutation_gene and mut_condition:
            target_symbol_result = duckdb.sql(f"""
                SELECT DISTINCT hugo_symbol 
                FROM '{path}/fact_mutations.csv'
                WHERE ensembl_id = '{target_gene}'
                LIMIT 1
            """).df()["hugo_symbol"].values
            target_symbol = target_symbol_result[0] if len(target_symbol_result) > 0 else target_gene

            mut_info_target = duckdb.sql(f"""
                SELECT ach_id,
                       STRING_AGG(protein_change, ', ') as {target_symbol}_mutations,
                       MAX(CAST(is_damaging AS INT)) as {target_symbol}_has_damaging
                FROM '{path}/fact_mutations.csv'
                WHERE ensembl_id = '{target_gene}'
                GROUP BY ach_id
            """).df().rename(columns={"ach_id": "ACH_ID"})

            mut_info_filter = duckdb.sql(f"""
                SELECT ach_id,
                       STRING_AGG(protein_change, ', ') as {mutation_gene}_mutations,
                       MAX(CAST(is_damaging AS INT)) as {mutation_gene}_has_damaging
                FROM '{path}/fact_mutations.csv'
                WHERE hugo_symbol = '{mutation_gene}'
                AND {mut_condition}
                GROUP BY ach_id
            """).df().rename(columns={"ach_id": "ACH_ID"})

            df_clean = df_clean.merge(mut_info_target, on="ACH_ID", how="left")
            df_clean = df_clean.merge(mut_info_filter, on="ACH_ID", how="left")

        # save to global + display
        df_result = df_clean.copy()
        display_result()

run_button.on_click(run_ranking)

Loading options...
Ready. 18212 mutation genes | 17498 fusion genes | 30 lineages | 49 diseases | 225 metabolites
